# AI-Based Fault Detection / Parameter Quantification on Engine Sensor Data

Walkthrough of the pipeline on the NASA C-MAPSS turbofan degradation dataset (FD001):
1. Load & prepare data (sliding windows, time-respecting split)
2. Train a lightweight DNN surrogate to estimate Remaining Useful Life (RUL)
3. Layer fault detection on top (Isolation Forest, rule-based, surrogate-threshold, hybrid)
4. Ablation: exact/physics-style baseline vs. surrogate-only vs. surrogate+rules
5. Evaluate: RMSE/NMSE for RUL, precision/recall/F1 for fault detection, accuracy-vs-cost plot

In [ ]:
import sys
sys.path.append('..')

from src.pipeline import run

summary = run(data_dir='../CMAPSSData', subset='FD001', results_dir='../results', epochs=40)
summary

## Inspect saved figures

- `results/figures/FD001_training_curves.png` — surrogate train/val loss
- `results/figures/FD001_rul_predictions.png` — predicted vs. true RUL scatter
- `results/figures/FD001_accuracy_vs_cost.png` — ablation: accuracy vs. inference cost

In [ ]:
from IPython.display import Image, display

for fig in ['FD001_training_curves.png', 'FD001_rul_predictions.png', 'FD001_accuracy_vs_cost.png']:
    display(Image(filename=f'../results/figures/{fig}'))

## Step-by-step (optional): rerun individual pieces

The cells below mirror `src/pipeline.py` but run each stage separately, useful for
experimenting with window size, model architecture, or detector thresholds.

In [ ]:
from src.data import load_dataset, drop_constant_sensors, clip_rul, COLUMN_NAMES
from src.windows import WindowConfig, make_windows, last_window_per_unit, unit_train_val_split
from src.surrogate import MLPSurrogate, BiLSTMSurrogate, train_surrogate, predict
from src.anomaly import IsolationForestDetector, RuleBasedDetector, rul_to_fault_label
from src.evaluate import rmse, nmse, cmapss_score, detection_metrics
from src.plots import plot_training_curves, plot_rul_predictions

ds = load_dataset('../CMAPSSData', 'FD001')
ds.train.head()

In [ ]:
# Try a BiLSTM surrogate instead of the MLP, as an extension.
sensor_cols = [c for c in COLUMN_NAMES if c.startswith('sensor_')]
active_sensors = drop_constant_sensors(ds.train, sensor_cols)
feature_cols = ['op_setting_1', 'op_setting_2', 'op_setting_3'] + active_sensors

feat_mean = ds.train[feature_cols].mean()
feat_std = ds.train[feature_cols].std() + 1e-8
train_df = ds.train.copy()
train_df[feature_cols] = (train_df[feature_cols] - feat_mean) / feat_std
train_df['RUL_clipped'] = clip_rul(train_df['RUL'])

cfg = WindowConfig(window_size=30, stride=1)
X_all, y_all, units_all = make_windows(train_df, feature_cols, 'RUL_clipped', config=cfg)
train_mask, val_mask = unit_train_val_split(units_all)

lstm = BiLSTMSurrogate(n_features=len(feature_cols))
result = train_surrogate(lstm, X_all[train_mask], y_all[train_mask], X_all[val_mask], y_all[val_mask], epochs=20)
plot_training_curves(result.train_losses, result.val_losses, title='BiLSTM surrogate training')